Author: Alana Pooler
<br>
Purpose: Complete Homework 10

In [1]:
from pyspark.sql.functions import sqrt, col
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 16:18:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 16:18:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Part 1: Creating Streaming Data Using `rate`

First, we will set up a data stream using the `rate` format.

In [2]:
df = spark \
    .readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

Before starting the stream, we want to do a couple transformations:
* Find the square root of the rate 'value'
* Find mod 4 of the rate 'value'

In [3]:
df_transformed = df.select(
    col("timestamp"),
    col("value"),
    sqrt(col("value")).alias("sqrt_value"),
    (col("value") % 4).alias("mod_4")
)

Now, we can start the stream and apply the transformations. We will use .format("memory") to write the output to memory.

In [8]:
query = df_transformed.writeStream \
    .format("memory") \
    .queryName("rate_table") \
    .outputMode("append") \
    .start()

26/04/21 16:23:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-05fba774-2bf9-40f5-a8ac-42ca69d94a9e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:23:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


After about 30 seconds, we will stop the query and look at the results.

In [9]:
query.stop()
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+-----+
|           timestamp|value|        sqrt_value|mod_4|
+--------------------+-----+------------------+-----+
|2026-04-21 16:23:...|    0|               0.0|    0|
|2026-04-21 16:23:...|    1|               1.0|    1|
|2026-04-21 16:23:...|    2|1.4142135623730951|    2|
|2026-04-21 16:23:...|    3|1.7320508075688772|    3|
|2026-04-21 16:23:...|    4|               2.0|    0|
|2026-04-21 16:23:...|    5|  2.23606797749979|    1|
|2026-04-21 16:23:...|    6| 2.449489742783178|    2|
|2026-04-21 16:23:...|    7|2.6457513110645907|    3|
|2026-04-21 16:23:...|    8|2.8284271247461903|    0|
|2026-04-21 16:23:...|    9|               3.0|    1|
|2026-04-21 16:23:...|   10|3.1622776601683795|    2|
|2026-04-21 16:23:...|   11|   3.3166247903554|    3|
|2026-04-21 16:23:...|   12|3.4641016151377544|    0|
|2026-04-21 16:23:...|   13| 3.605551275463989|    1|
|2026-04-21 16:23:...|   14|3.7416573867739413|    2|
|2026-04-21 16:23:...|   15|

26/04/21 16:23:41 WARN DAGScheduler: Failed to cancel job group e11d8208-fa03-4a7d-94e0-77608616c5e0. Cannot find active jobs for it.
26/04/21 16:23:41 WARN DAGScheduler: Failed to cancel job group e11d8208-fa03-4a7d-94e0-77608616c5e0. Cannot find active jobs for it.
